# CMPE 58T — 05b: Morphological Feature Injection V3 (Zemberek)

Built directly on `phase5_morph_injection_zemberek_v3.ipynb`.
Additions: `PRED_DIR`/`RESULTS_DIR`, checkpoint skip logic, fixed morph state save,
npy saving for McNemar's tests, per-relation LAS, V2 comparison table, verify cell.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR  = "/content/drive/MyDrive/58T - Application Project"
DATA_DIR     = os.path.join(PROJECT_DIR, "data")
VOCAB_DIR    = os.path.join(PROJECT_DIR, "vocab")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
MORPH_DIR    = os.path.join(MODEL_DIR, "morph")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache")
PRED_DIR     = os.path.join(PROJECT_DIR, "predictions")
RESULTS_DIR  = os.path.join(PROJECT_DIR, "results")

for d in [DATA_DIR, VOCAB_DIR, MODEL_DIR, MORPH_DIR, CACHE_DIR, PRED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

for tb in ["BOUN", "IMST", "PUD"]:
    path = os.path.join(DATA_DIR, f"UD_Turkish-{tb}")
    print(f"  {'OK' if os.path.isdir(path) else 'MISSING'}  UD_Turkish-{tb}")

print(f"  {'OK' if os.path.exists(os.path.join(VOCAB_DIR, 'feat_vocab.json')) else 'MISSING'}  feat_vocab.json")


Mounted at /content/drive
  OK  UD_Turkish-BOUN
  OK  UD_Turkish-IMST
  OK  UD_Turkish-PUD
  OK  feat_vocab.json


In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "transformers==4.35.2" -q
!pip install supar -q
!pip install conllu -q
!pip install zemberek-python -q

import os

PARSER_PY = "/usr/local/lib/python3.12/dist-packages/supar/parsers/parser.py"
OLD = "state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload))"
NEW = "state = torch.load(path if os.path.exists(path) else download(supar.MODEL[src].get(path, path), reload=reload), weights_only=False)"
with open(PARSER_PY, "r") as f:
    src = f.read()
if OLD in src:
    with open(PARSER_PY, "w") as f:
        f.write(src.replace(OLD, NEW))
    print("Patch 1 applied : torch.load weights_only=False")
else:
    print("Patch 1 OK      : already patched")

from transformers import AdamW, get_linear_schedule_with_warmup
print("Patch 2 OK      : transformers.AdamW available")

import torch, transformers
print(f"\ntransformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU — switch runtime before training")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 45.9 MB/

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Patch 2 OK      : transformers.AdamW available

transformers : 4.35.2
PyTorch      : 2.10.0+cu128
CUDA         : True
GPU          : NVIDIA A100-SXM4-40GB


In [3]:
import os, glob, json
import torch

BERTURK_NAME   = "dbmdz/bert-base-turkish-uncased"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

UD_MORPH_DIM   = 32
ZEMB_MORPH_DIM = 32
MAX_UD_FEATS   = 16
MAX_ZEMB_FEATS = 10

HIDDEN_SIZE  = 800
DROPOUT      = 0.33
BATCH_SIZE   = 5000
MAX_EPOCHS   = 10
PATIENCE     = 5
LR           = 2e-3
BERT_LR      = 2e-5
WARMUP       = 0.1
WEIGHT_DECAY = 0.01
EPS          = 1e-8
GRAD_CLIP    = 5.0

SUPAR_EXTRA = dict(
    mbr=True, partial=False, tree=False,
    proj=False, punct=False, update_steps=1, clip=GRAD_CLIP
)

def get_splits(tb_name):
    folder = os.path.join(DATA_DIR, f"UD_Turkish-{tb_name}")
    files  = sorted(glob.glob(os.path.join(folder, "*.conllu")))
    splits = {}
    for f in files:
        name = os.path.basename(f)
        if   "train" in name: splits["train"] = f
        elif "dev"   in name: splits["dev"]   = f
        elif "test"  in name: splits["test"]  = f
    return splits

BOUN = get_splits("BOUN")
IMST = get_splits("IMST")
PUD  = get_splits("PUD")

MODEL_V1       = os.path.join(MODEL_DIR,  "baseline_boun.pt")        # from 02_baseline
MODEL_V2       = os.path.join(MORPH_DIR,  "morph_ud_feats_boun.pt")
MODEL_V3       = os.path.join(MORPH_DIR,  "morph_zemberek_boun.pt")
MORPH_STATE_V3 = MODEL_V3 + ".morph_state.pt"
ZEMB_CACHE     = os.path.join(CACHE_DIR,  "zemberek_features.json")
TEXT_CACHE_PATH = os.path.join(CACHE_DIR, "zemberek_features_by_text.json")

print(f"Device : {DEVICE}")
print(f"MODEL_V1 : {'exists' if os.path.exists(MODEL_V1) else 'MISSING'}")
print(f"MODEL_V2 : {'exists' if os.path.exists(MODEL_V2) else 'missing'}")
print(f"MODEL_V3 : {'exists' if os.path.exists(MODEL_V3) else 'new'}")
print(f"Zemb cache : {'exists' if os.path.exists(TEXT_CACHE_PATH) else 'will be built'}")


Device : cuda
MODEL_V1 : exists
MODEL_V2 : exists
MODEL_V3 : new
Zemb cache : exists


In [4]:
import json

feat_vocab_path = os.path.join(VOCAB_DIR, "feat_vocab.json")
with open(feat_vocab_path, "r", encoding="utf-8") as f:
    UD_FEAT_VOCAB = json.load(f)

UD_PAD_IDX   = UD_FEAT_VOCAB["<PAD>"]
UD_UNK_IDX   = UD_FEAT_VOCAB["<UNK>"]
UD_VOCAB_SIZE = len(UD_FEAT_VOCAB)

print(f"UD FEATS vocabulary: {UD_VOCAB_SIZE} entries  (PAD={UD_PAD_IDX}, UNK={UD_UNK_IDX})")


UD FEATS vocabulary: 87 entries  (PAD=85, UNK=86)


In [5]:
import conllu
from zemberek import TurkishMorphology

def get_zemb_features(token_form, morphology):
    try:
        result = morphology.analyze(token_form)
        if result.analysis_results:
            return [f"Zemb:{m.id_}" for m in result.analysis_results[0].get_morphemes()]
    except Exception:
        pass
    return ["Zemb:Unknown"]


def precompute_zemberek(splits_dict, morphology, cache_path):
    if os.path.exists(cache_path):
        print(f"Loading Zemberek cache from {cache_path}")
        with open(cache_path, "r", encoding="utf-8") as f:
            cache = json.load(f)
        print(f"  Cache loaded: {len(cache)} entries")
        return cache

    print("Building Zemberek cache (runs once, saved to Drive)...")
    cache = {}
    total = 0

    for tb_name, splits in splits_dict.items():
        for split_name, path in splits.items():
            print(f"  Analyzing {tb_name} {split_name}: {path}")
            with open(path, "r", encoding="utf-8") as f:
                sentences = conllu.parse(f.read())
            for sent_idx, sent in enumerate(sentences):
                key = f"{tb_name}:{split_name}:{sent_idx}"
                token_features = []
                for tok in sent:
                    if not isinstance(tok["id"], int):
                        continue
                    feats = get_zemb_features(tok["form"], morphology)
                    token_features.append(feats)
                    total += 1
                cache[key] = token_features
            print(f"    {len(sentences)} sentences processed")

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)
    print(f"\nCache saved: {len(cache)} sentences, {total} tokens -> {cache_path}")
    return cache


print("Initializing Zemberek morphological analyzer...")
import logging
logging.getLogger('zemberek').setLevel(logging.WARNING)
morphology = TurkishMorphology.create_with_defaults()
print("Zemberek ready.")

splits_to_analyze = {
    "BOUN": BOUN,
    "PUD":  {"test": PUD["test"]},
}
ZEMB_CACHE_DATA = precompute_zemberek(splits_to_analyze, morphology, ZEMB_CACHE)


Initializing Zemberek morphological analyzer...
Zemberek ready.
Loading Zemberek cache from /content/drive/MyDrive/58T - Application Project/cache/zemberek_features.json
  Cache loaded: 10761 entries


In [6]:
from collections import Counter

zemb_vocab_path = os.path.join(VOCAB_DIR, "zemb_vocab.json")

if os.path.exists(zemb_vocab_path):
    with open(zemb_vocab_path, "r", encoding="utf-8") as f:
        ZEMB_VOCAB = json.load(f)
    print(f"Loaded Zemberek vocab: {len(ZEMB_VOCAB)} entries")
else:
    zemb_counts = Counter()
    for token_features_list in ZEMB_CACHE_DATA.values():
        for token_feats in token_features_list:
            for feat in token_feats:
                zemb_counts[feat] += 1
    ZEMB_VOCAB = {feat: idx for idx, (feat, _) in enumerate(zemb_counts.most_common())}
    ZEMB_VOCAB["<PAD>"] = len(ZEMB_VOCAB)
    ZEMB_VOCAB["<UNK>"] = len(ZEMB_VOCAB)
    with open(zemb_vocab_path, "w", encoding="utf-8") as f:
        json.dump(ZEMB_VOCAB, f, ensure_ascii=False, indent=2)
    print(f"Built Zemberek vocab: {len(ZEMB_VOCAB)} entries")

ZEMB_PAD_IDX    = ZEMB_VOCAB["<PAD>"]
ZEMB_UNK_IDX    = ZEMB_VOCAB["<UNK>"]
ZEMB_VOCAB_SIZE = len(ZEMB_VOCAB)
print(f"  Vocab size : {ZEMB_VOCAB_SIZE}  PAD={ZEMB_PAD_IDX}  UNK={ZEMB_UNK_IDX}")


Loaded Zemberek vocab: 88 entries
  Vocab size : 88  PAD=86  UNK=87


In [7]:
import torch
import torch.nn as nn
import types

class MorphEmbedder(nn.Module):
    def __init__(self, vocab_size, morph_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, morph_dim, padding_idx=pad_idx)
        nn.init.normal_(self.embedding.weight, std=0.01)
        with torch.no_grad():
            self.embedding.weight[pad_idx].fill_(0)
    def forward(self, indices):
        return self.embedding(indices).sum(dim=2)

class V3MorphProjection(nn.Module):
    def __init__(self, n_hidden, ud_dim, zemb_dim):
        super().__init__()
        self.proj = nn.Linear(n_hidden + ud_dim + zemb_dim, n_hidden)
        nn.init.eye_(self.proj.weight[:n_hidden, :n_hidden])
        nn.init.zeros_(self.proj.weight[:, n_hidden:])
        nn.init.zeros_(self.proj.bias)
    def forward(self, bert_out, ud_emb, zemb_emb):
        return self.proj(torch.cat([bert_out, ud_emb, zemb_emb], dim=-1))

FEATS_COL = 5

def batch_to_ud_tensor(batch, ud_vocab, ud_pad_idx, ud_unk_idx, max_feats, device):
    batch_size = len(batch.sentences)
    seq_len    = max(len(s.values[FEATS_COL]) + 1 for s in batch.sentences)
    morph      = torch.full((batch_size, seq_len, max_feats), ud_pad_idx, dtype=torch.long)
    for b, sent in enumerate(batch.sentences):
        for t, feat_str in enumerate(sent.values[FEATS_COL], start=1):
            if feat_str == "_" or not feat_str: continue
            for f_idx, pair in enumerate(feat_str.split("|")[:max_feats]):
                morph[b, t, f_idx] = ud_vocab.get(pair.strip(), ud_unk_idx)
    return morph.to(device)

def batch_to_zemb_tensor(batch, zemb_cache, zemb_vocab, zemb_pad_idx,
                         zemb_unk_idx, max_feats, device, split_key):
    batch_size = len(batch.sentences)
    seq_len    = max(len(s.values[FEATS_COL]) + 1 for s in batch.sentences)
    morph      = torch.full((batch_size, seq_len, max_feats), zemb_pad_idx, dtype=torch.long)
    for b, sent in enumerate(batch.sentences):
        sent_text = sent.metadata.get("text", "") if hasattr(sent, "metadata") else ""
        cache_key = f"{split_key}:{sent_text}"
        token_feats = zemb_cache.get(cache_key, None)
        if token_feats is None: continue
        for t, feat_list in enumerate(token_feats, start=1):
            for f_idx, feat in enumerate(feat_list[:max_feats]):
                morph[b, t, f_idx] = zemb_vocab.get(feat, zemb_unk_idx)
    return morph.to(device)

def patch_model_forward_v3(parser, ud_embedder, zemb_embedder, projection):
    from supar.utils.common import MIN
    _ud_emb = ud_embedder; _zemb_emb = zemb_embedder
    _proj = projection; _encode = parser.model.encode

    def forward_v3(self, words, feats=None, ud_indices=None, zemb_indices=None):
        x = _encode(words, feats)
        if ud_indices is not None and zemb_indices is not None:
            x = _proj(x, _ud_emb(ud_indices), _zemb_emb(zemb_indices))
        x = self.encoder_dropout(x)
        mask  = words.ne(self.args.pad_index) if len(words.shape) < 3 else words.ne(self.args.pad_index).any(-1)
        arc_d = self.arc_mlp_d(x);  arc_h = self.arc_mlp_h(x)
        sib_s = self.sib_mlp_s(x);  sib_d = self.sib_mlp_d(x);  sib_h = self.sib_mlp_h(x)
        rel_d = self.rel_mlp_d(x);  rel_h = self.rel_mlp_h(x)
        s_arc = self.arc_attn(arc_d, arc_h).masked_fill_(~mask.unsqueeze(1), MIN)
        s_sib = self.sib_attn(sib_s, sib_d, sib_h).permute(0, 3, 1, 2)
        s_rel = self.rel_attn(rel_d, rel_h).permute(0, 2, 3, 1)
        return s_arc, s_sib, s_rel

    parser.model.forward = types.MethodType(forward_v3, parser.model)
    print("Model forward patched for V3.")

print("V3 modules defined.")


V3 modules defined.


In [8]:
import conllu

if os.path.exists(TEXT_CACHE_PATH):
    print(f"Loading text-keyed Zemberek cache...")
    with open(TEXT_CACHE_PATH, "r", encoding="utf-8") as f:
        ZEMB_TEXT_CACHE = json.load(f)
    print(f"  {len(ZEMB_TEXT_CACHE)} entries loaded")
else:
    print("Building text-keyed Zemberek cache...")
    ZEMB_TEXT_CACHE = {}
    total = 0
    all_paths = [
        ("BOUN:train", BOUN["train"]), ("BOUN:dev", BOUN["dev"]),
        ("BOUN:test",  BOUN["test"]),  ("PUD:test", PUD["test"]),
    ]
    for split_key, path in all_paths:
        with open(path, "r", encoding="utf-8") as f:
            sentences = conllu.parse(f.read())
        for sent in sentences:
            sent_text = sent.metadata.get("text", "") if sent.metadata else ""
            if not sent_text:
                sent_text = " ".join(tok["form"] for tok in sent if isinstance(tok["id"], int))
            cache_key = f"{split_key}:{sent_text}"
            token_features = []
            for tok in sent:
                if not isinstance(tok["id"], int): continue
                feats = get_zemb_features(tok["form"], morphology)
                token_features.append(feats)
                total += 1
            ZEMB_TEXT_CACHE[cache_key] = token_features
        print(f"  {split_key}: {len(sentences)} sentences")
    with open(TEXT_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(ZEMB_TEXT_CACHE, f, ensure_ascii=False)
    print(f"\nText cache saved: {len(ZEMB_TEXT_CACHE)} sentences, {total} tokens")


Loading text-keyed Zemberek cache...
  10701 entries loaded


In [9]:
from supar import CRF2oDependencyParser

if os.path.exists(MODEL_V3):
    print(f"V3 checkpoint found — loading from {MODEL_V3}")
    parser = CRF2oDependencyParser.load(MODEL_V3)
elif os.path.exists(MODEL_V1):
    print(f"Loading baseline_boun.pt as starting point: {MODEL_V1}")
    parser = CRF2oDependencyParser.load(MODEL_V1)
else:
    raise FileNotFoundError(f"MODEL_V1 not found: {MODEL_V1}. Run 02_baseline.ipynb first.")

parser.args.update(SUPAR_EXTRA)
parser.model = parser.model.to(DEVICE)
n_hidden = parser.model.encoder.n_out

ud_embedder   = MorphEmbedder(UD_VOCAB_SIZE,   UD_MORPH_DIM,   UD_PAD_IDX).to(DEVICE)
zemb_embedder = MorphEmbedder(ZEMB_VOCAB_SIZE, ZEMB_MORPH_DIM, ZEMB_PAD_IDX).to(DEVICE)
projection    = V3MorphProjection(n_hidden, UD_MORPH_DIM, ZEMB_MORPH_DIM).to(DEVICE)

# Load saved morph weights if resuming from V3 checkpoint
if os.path.exists(MORPH_STATE_V3):
    state = torch.load(MORPH_STATE_V3, map_location=DEVICE, weights_only=False)
    ud_embedder.load_state_dict(state["ud_embedder"])
    zemb_embedder.load_state_dict(state["zemb_embedder"])
    projection.load_state_dict(state["projection"])
    print("V3 morph module weights loaded from checkpoint.")

patch_model_forward_v3(parser, ud_embedder, zemb_embedder, projection)

morph_params = (sum(p.numel() for p in ud_embedder.parameters()) +
                sum(p.numel() for p in zemb_embedder.parameters()) +
                sum(p.numel() for p in projection.parameters()))
print(f"  n_hidden={n_hidden}  morph_params={morph_params:,}  device={DEVICE}")


INFO:numexpr.utils:NumExpr defaulting to 12 threads.


2026-05-26 11:40:50,405 - numexpr.utils - INFO
Msg: NumExpr defaulting to 12 threads.

Loading baseline_boun.pt as starting point: /content/drive/MyDrive/58T - Application Project/models/baseline_boun.pt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Model forward patched for V3.
  n_hidden=768  morph_params=645,344  device=cuda


In [10]:
from supar.utils import Dataset
from transformers import AdamW, get_linear_schedule_with_warmup

def make_loader(parser, path, batch_size, shuffle=True):
    dataset = Dataset(parser.transform, path)
    dataset.build(batch_size, 32)
    return dataset.loader

print("Building data loaders...")
train_loader = make_loader(parser, BOUN["train"], BATCH_SIZE, shuffle=True)
dev_loader   = make_loader(parser, BOUN["dev"],   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(parser, BOUN["test"],  BATCH_SIZE, shuffle=False)
pud_loader   = make_loader(parser, PUD["test"],   BATCH_SIZE, shuffle=False)
print(f"  train={len(train_loader)}  dev={len(dev_loader)}  test={len(test_loader)}  pud={len(pud_loader)} batches")

total_steps  = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP)

bert_params = list(parser.model.encoder.parameters())
task_params = (
    [p for p in parser.model.parameters() if not any(p is bp for bp in bert_params)] +
    list(ud_embedder.parameters()) + list(zemb_embedder.parameters()) +
    list(projection.parameters())
)
all_params = bert_params + task_params

optimizer = AdamW([{"params": bert_params, "lr": BERT_LR},
                   {"params": task_params, "lr": LR}],
                  weight_decay=WEIGHT_DECAY, eps=EPS)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f"  total_steps={total_steps}  warmup={warmup_steps}")


Building data loaders...


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


  train=39  dev=32  test=32  pud=32 batches


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  total_steps=390  warmup=39


In [11]:
from supar.utils.metric import AttachmentMetric

def run_epoch_v3(parser, loader, ud_embedder, zemb_embedder, projection,
                 optimizer, scheduler, device, split_key, training=True):
    if training:
        parser.model.train(); ud_embedder.train()
        zemb_embedder.train(); projection.train()
    else:
        parser.model.eval(); ud_embedder.eval()
        zemb_embedder.eval(); projection.eval()

    total_loss = 0.0
    metric     = AttachmentMetric()
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for batch in loader:
            words, texts, *feats, arcs, sibs, rels = batch
            word_mask = words.ne(parser.args.pad_index)
            mask      = word_mask if len(words.shape) < 3 else word_mask.any(-1)
            mask[:, 0] = 0
            words = words.to(device); feats = [f.to(device) for f in feats]
            arcs  = arcs.to(device);  sibs  = sibs.to(device)
            rels  = rels.to(device);  mask  = mask.to(device)

            ud_indices = batch_to_ud_tensor(
                batch, UD_FEAT_VOCAB, UD_PAD_IDX, UD_UNK_IDX, MAX_UD_FEATS, device)
            ud_indices = ud_indices[:, :words.shape[1], :]

            zemb_indices = batch_to_zemb_tensor(
                batch, ZEMB_TEXT_CACHE, ZEMB_VOCAB, ZEMB_PAD_IDX,
                ZEMB_UNK_IDX, MAX_ZEMB_FEATS, device, split_key)
            zemb_indices = zemb_indices[:, :words.shape[1], :]

            s_arc, s_sib, s_rel = parser.model(words, feats, ud_indices, zemb_indices)
            loss, s_arc = parser.model.loss(
                s_arc, s_sib, s_rel, arcs, sibs, rels, mask,
                parser.args.mbr, parser.args.partial)

            if training:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(all_params, GRAD_CLIP)
                optimizer.step(); scheduler.step()

            arc_preds, rel_preds = parser.model.decode(
                s_arc, s_sib, s_rel, mask,
                parser.args.tree, parser.args.mbr, parser.args.proj)
            total_loss += loss.item()
            metric(arc_preds, rel_preds, arcs, rels, mask)

    return total_loss / len(loader), metric


if os.path.exists(MODEL_V3) and os.path.exists(MORPH_STATE_V3):
    print("V3 checkpoint exists — skipping training.")
    print("Delete morph_zemberek_boun.pt and .morph_state.pt to retrain.")
else:
    print(f"Training V3 | Epochs={MAX_EPOCHS}  Patience={PATIENCE}  Device={DEVICE}")
    print("=" * 65)

    best_las = 0.0; patience_ct = 0; history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, _ = run_epoch_v3(
            parser, train_loader, ud_embedder, zemb_embedder, projection,
            optimizer, scheduler, DEVICE, "BOUN:train", training=True)
        dev_loss, dev_m = run_epoch_v3(
            parser, dev_loader, ud_embedder, zemb_embedder, projection,
            optimizer, scheduler, DEVICE, "BOUN:dev", training=False)

        uas = dev_m.uas * 100; las = dev_m.las * 100
        print(f"Epoch {epoch:>2}/{MAX_EPOCHS}  train {train_loss:.4f} | dev {dev_loss:.4f} | UAS {uas:.2f} LAS {las:.2f}", end="")

        if las > best_las:
            best_las = las; patience_ct = 0
            torch.save({"ud_embedder":   ud_embedder.state_dict(),
                        "zemb_embedder": zemb_embedder.state_dict(),
                        "projection":    projection.state_dict()},
                       MORPH_STATE_V3)
            parser.save(MODEL_V3)
            print(" (saved)", end="")
        else:
            patience_ct += 1
            print(f" (patience {patience_ct}/{PATIENCE})", end="")
        print()
        history.append({"epoch": epoch, "train_loss": train_loss,
                        "dev_loss": dev_loss, "dev_uas": uas, "dev_las": las})
        if patience_ct >= PATIENCE:
            print(f"Early stopping at epoch {epoch}."); break

    print(f"\nBest dev LAS: {best_las:.2f}")
    with open(os.path.join(RESULTS_DIR, "morph_v3_history.json"), "w") as f:
        json.dump(history, f, indent=2)
    print("History saved.")


Training V3 | Epochs=10  Patience=5  Device=cuda


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch  1/10  train 219.6193 | dev 34.6962 | UAS 62.12 LAS 23.69 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch  2/10  train 6.3252 | dev 1.9647 | UAS 71.05 LAS 54.80 (saved)


...

Epoch  8/10  train 0.6538 | dev 1.7147 | UAS 80.05 LAS 71.60 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch  9/10  train 0.6005 | dev 1.9031 | UAS 80.14 LAS 71.76 (saved)


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


Epoch 10/10  train 0.5646 | dev 1.8529 | UAS 80.17 LAS 72.06 (saved)

Best dev LAS: 72.06
History saved.


In [12]:
import json, numpy as np
import conllu as conllu_lib
from collections import defaultdict

# Load best V3 morph state
if os.path.exists(MORPH_STATE_V3):
    state = torch.load(MORPH_STATE_V3, map_location=DEVICE, weights_only=False)
    ud_embedder.load_state_dict(state["ud_embedder"])
    zemb_embedder.load_state_dict(state["zemb_embedder"])
    projection.load_state_dict(state["projection"])
    print("Best V3 morph state loaded.")

def evaluate_v3(loader, split_key, label):
    _, metric = run_epoch_v3(parser, loader, ud_embedder, zemb_embedder, projection,
                              None, None, DEVICE, split_key, training=False)
    r = {"system": label,
         "UAS": round(metric.uas * 100, 2),
         "LAS": round(metric.las * 100, 2)}
    print(f"  {label:<45} UAS {r['UAS']:>6.2f}   LAS {r['LAS']:>6.2f}")
    return r

def per_relation_las(gold_sents, pred_sents):
    counts = defaultdict(lambda: [0, 0])
    for gold, pred in zip(gold_sents, pred_sents):
        gt = [t for t in gold if isinstance(t["id"], int)]
        pt = [t for t in pred if isinstance(t["id"], int)]
        for g, p in zip(gt, pt):
            counts[g["deprel"]][1] += 1
            if g["head"] == p["head"] and g["deprel"] == p["deprel"]:
                counts[g["deprel"]][0] += 1
    return {rel: {"correct": c, "total": t,
                  "LAS": round(100 * c / t, 2) if t else 0.0}
            for rel, (c, t) in sorted(counts.items(), key=lambda x: -x[1][1])}

results = []
print("=" * 70)

# V3 on BOUN test and PUD
r = evaluate_v3(test_loader, "BOUN:test", "V3: BERTurk + UD FEATS + Zemberek")
r["test"] = "BOUN"; results.append(r)
r = evaluate_v3(pud_loader,  "PUD:test",  "V3: BERTurk + UD FEATS + Zemberek -> PUD")
r["test"] = "PUD";  results.append(r)

# Save conllu + npy for McNemar's tests
for sys_name, test_name, conllu_path, split_key in [
        ("morph_v3_zemberek", "BOUN", BOUN["test"], "BOUN:test"),
        ("morph_v3_zemberek", "PUD",  PUD["test"],  "PUD:test")]:
    pred_conllu = os.path.join(PRED_DIR, f"{sys_name}_{test_name}.conllu")
    parser.predict(conllu_path, pred=pred_conllu, verbose=False)
    with open(conllu_path, encoding="utf-8") as f:
        gold_sents = conllu_lib.parse(f.read())
    with open(pred_conllu, encoding="utf-8") as f:
        pred_sents = conllu_lib.parse(f.read())
    sent_las = [
        sum(1 for g, p in zip(
            [t for t in gold if isinstance(t["id"], int)],
            [t for t in pred if isinstance(t["id"], int)])
            if g["head"] == p["head"] and g["deprel"] == p["deprel"]) ==
        len([t for t in gold if isinstance(t["id"], int)])
        for gold, pred in zip(gold_sents, pred_sents)
    ]
    np.save(os.path.join(PRED_DIR, f"{sys_name}_{test_name}_sent_las.npy"),
            np.array(sent_las, dtype=bool))
    print(f"  Saved: {sys_name}_{test_name}.conllu + _sent_las.npy")

# Per-relation LAS on BOUN test
pred_boun = os.path.join(PRED_DIR, "morph_v3_zemberek_BOUN.conllu")
if os.path.exists(pred_boun):
    with open(BOUN["test"], encoding="utf-8") as f:
        gold_boun = conllu_lib.parse(f.read())
    with open(pred_boun, encoding="utf-8") as f:
        pred_boun_sents = conllu_lib.parse(f.read())
    breakdown = per_relation_las(gold_boun, pred_boun_sents)
    br_path = os.path.join(RESULTS_DIR, "morph_v3_BOUN_per_rel.json")
    with open(br_path, "w") as f:
        json.dump(breakdown, f, indent=2)
    print(f"  Per-relation saved: {br_path}")

# Save results JSON
results_path = os.path.join(RESULTS_DIR, "morph_v3_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {results_path}")


Best V3 morph state loaded.


/usr/local/lib/python3.12/dist-packages/supar/utils/fn.py:155: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out_tensor[i][[slice(-i, None) if padding_side == 'left' else slice(0, i) for i in tensor.size()]] = tensor


  V3: BERTurk + UD FEATS + Zemberek             UAS  80.15   LAS  71.79
  V3: BERTurk + UD FEATS + Zemberek -> PUD      UAS  76.23   LAS  63.65
  Saved: morph_v3_zemberek_BOUN.conllu + _sent_las.npy
  Saved: morph_v3_zemberek_PUD.conllu + _sent_las.npy
  Per-relation saved: /content/drive/MyDrive/58T - Application Project/results/morph_v3_BOUN_per_rel.json

Results saved: /content/drive/MyDrive/58T - Application Project/results/morph_v3_results.json


In [13]:
# Load V2 results for comparison
v2_path = os.path.join(RESULTS_DIR, "morph_v2_results.json")
v2_results = []
if os.path.exists(v2_path):
    with open(v2_path) as f:
        v2_results = json.load(f)

print()
print(f"{'System':<47} {'Test':<6} {'UAS':>7} {'LAS':>7}")
print("-" * 72)
for r in v2_results:
    print(f"  {r['system']:<45} {r['test']:<6} {r['UAS']:>7.2f} {r['LAS']:>7.2f}")
for r in results:
    print(f"  {r['system']:<45} {r['test']:<6} {r['UAS']:>7.2f} {r['LAS']:>7.2f}")

v2_boun = next((r for r in v2_results if "V2" in r.get("system","") and r["test"]=="BOUN"), None)
v3_boun = next((r for r in results if r["test"]=="BOUN"), None)
if v2_boun and v3_boun:
    d = v3_boun["LAS"] - v2_boun["LAS"]
    print(f"\nDelta V3 - V2 (BOUN): ΔLAS {d:+.2f}")



System                                          Test       UAS     LAS
------------------------------------------------------------------------
  V1: BERTurk only (baseline)                   BOUN     76.63   63.91
  V1: BERTurk only -> PUD                       PUD      73.71   59.86
  V2: BERTurk + UD FEATS                        BOUN     79.92   71.79
  V2: BERTurk + UD FEATS -> PUD                 PUD      76.43   64.05
  V3: BERTurk + UD FEATS + Zemberek             BOUN     80.15   71.79
  V3: BERTurk + UD FEATS + Zemberek -> PUD      PUD      76.23   63.65

Delta V3 - V2 (BOUN): ΔLAS +0.00


In [14]:
expected = [
    (MORPH_DIR,   "morph_zemberek_boun.pt"),
    (MORPH_DIR,   "morph_zemberek_boun.pt.morph_state.pt"),
    (VOCAB_DIR,   "zemb_vocab.json"),
    (CACHE_DIR,   "zemberek_features_by_text.json"),
    (PRED_DIR,    "morph_v3_zemberek_BOUN.conllu"),
    (PRED_DIR,    "morph_v3_zemberek_BOUN_sent_las.npy"),
    (PRED_DIR,    "morph_v3_zemberek_PUD.conllu"),
    (PRED_DIR,    "morph_v3_zemberek_PUD_sent_las.npy"),
    (RESULTS_DIR, "morph_v3_results.json"),
    (RESULTS_DIR, "morph_v3_history.json"),
    (RESULTS_DIR, "morph_v3_BOUN_per_rel.json"),
]

print(f"{'Status':<8} {'Size':>12}  {'Dir':<14} File")
print("-" * 80)
all_ok = True
for directory, filename in expected:
    path   = os.path.join(directory, filename)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path):,} B" if exists else "-"
    status = "OK" if exists else "MISSING"
    folder = os.path.basename(directory)
    if not exists: all_ok = False
    print(f"  {status:<6} {size:>12}  {folder:<14} {filename}")
print()
if all_ok:
    print("All files present. 05b complete.")
else:
    print("Some files missing — rerun relevant cells.")


Status           Size  Dir            File
--------------------------------------------------------------------------------
  OK     457,211,605 B  morph          morph_zemberek_boun.pt
  OK      2,584,427 B  morph          morph_zemberek_boun.pt.morph_state.pt
  OK          1,761 B  vocab          zemb_vocab.json
  OK      5,646,223 B  cache          zemberek_features_by_text.json
  OK        978,255 B  predictions    morph_v3_zemberek_BOUN.conllu
  OK          1,107 B  predictions    morph_v3_zemberek_BOUN_sent_las.npy
  OK      1,405,419 B  predictions    morph_v3_zemberek_PUD.conllu
  OK          1,128 B  predictions    morph_v3_zemberek_PUD_sent_las.npy
  OK            238 B  results        morph_v3_results.json
  OK          1,656 B  results        morph_v3_history.json
  OK          3,089 B  results        morph_v3_BOUN_per_rel.json

All files present. 05b complete.
